# Clothing Classification Pipeline

Preprocessing: CLAHE + RemBG + Localization + Cropping
Model: EfficientNet (Feature Extraction) + Classical ML (Classification)

## 1. Install Dependencies

In [ ]:
!pip install -q rembg opencv-python-headless scikit-image pillow transformers torch torchvision scikit-learn xgboost pandas matplotlib

## 2. Import Libraries

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
from rembg import remove
from skimage.restoration import denoise_bilateral
from skimage import img_as_ubyte, img_as_float
import torch
from transformers import AutoFeatureExtractor, AutoModel
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import xgboost as xgb
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

## 3. Load Dataset

In [ ]:
train_df = pd.read_csv('train.csv')
sample_submission = pd.read_csv('sample_submission.csv')

TRAIN_DIR = 'train/train'
TEST_DIR = 'test/test'

print(f"Training samples: {len(train_df)}")
print(f"Test samples: {len(sample_submission)}")
print(f"\nLabel distribution:")
print(f"Jenis: {train_df['jenis'].value_counts().to_dict()}")
print(f"Warna: {train_df['warna'].value_counts().to_dict()}")

## 4. Preprocessing Pipeline Functions

In [ ]:
def apply_clahe(image):
    """Apply CLAHE for adaptive contrast enhancement"""
    img_array = np.array(image)
    
    # Denoise
    img_float = img_as_float(img_array)
    denoised = denoise_bilateral(img_float, sigma_color=0.03, sigma_spatial=10, channel_axis=-1)
    denoised = img_as_ubyte(denoised)
    
    # CLAHE on L channel
    img_lab = cv2.cvtColor(denoised, cv2.COLOR_RGB2LAB)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img_lab[:, :, 0] = clahe.apply(img_lab[:, :, 0])
    enhanced = cv2.cvtColor(img_lab, cv2.COLOR_LAB2RGB)
    
    # Light sharpening
    kernel = np.array([[-0.5, -0.5, -0.5],
                       [-0.5,  5.0, -0.5],
                       [-0.5, -0.5, -0.5]])
    sharpened = cv2.filter2D(enhanced, -1, kernel)
    result = cv2.addWeighted(enhanced, 0.7, sharpened, 0.3, 0)
    
    return Image.fromarray(result)


def remove_background(image):
    """Remove background using rembg"""
    output = remove(image)
    return output


def get_bounding_box(image):
    """Get bounding box from alpha channel"""
    img_array = np.array(image)
    
    if img_array.shape[2] == 4:
        alpha = img_array[:, :, 3]
    else:
        gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
        _, alpha = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
    
    coords = cv2.findNonZero(alpha)
    if coords is None:
        return None
    
    x, y, w, h = cv2.boundingRect(coords)
    return (x, y, w, h)


def crop_with_padding(image, bbox, padding=0.05):
    """Crop image with padding"""
    if bbox is None:
        return image
    
    x, y, w, h = bbox
    img_array = np.array(image)
    height, width = img_array.shape[:2]
    
    pad_w = int(w * padding)
    pad_h = int(h * padding)
    
    x1 = max(0, x - pad_w)
    y1 = max(0, y - pad_h)
    x2 = min(width, x + w + pad_w)
    y2 = min(height, y + h + pad_h)
    
    cropped = img_array[y1:y2, x1:x2]
    
    # Convert to RGB if has alpha
    if cropped.shape[2] == 4:
        white_bg = np.ones((cropped.shape[0], cropped.shape[1], 3), dtype=np.uint8) * 255
        alpha = cropped[:, :, 3:4] / 255.0
        rgb = cropped[:, :, :3]
        cropped = (rgb * alpha + white_bg * (1 - alpha)).astype(np.uint8)
    
    return Image.fromarray(cropped)


def resize_image(image, target_size=(224, 224)):
    """Resize image maintaining aspect ratio"""
    img = image.copy()
    img.thumbnail(target_size, Image.Resampling.LANCZOS)
    
    # Pad to square
    new_img = Image.new('RGB', target_size, (255, 255, 255))
    paste_x = (target_size[0] - img.width) // 2
    paste_y = (target_size[1] - img.height) // 2
    new_img.paste(img, (paste_x, paste_y))
    
    return new_img


def preprocess_pipeline(image_path):
    """Complete preprocessing pipeline"""
    # Load image
    image = Image.open(image_path).convert('RGB')
    
    # Step 1: CLAHE enhancement
    enhanced = apply_clahe(image)
    
    # Step 2: Remove background
    no_bg = remove_background(enhanced)
    
    # Step 3: Get bounding box
    bbox = get_bounding_box(no_bg)
    
    # Step 4: Crop with padding
    cropped = crop_with_padding(no_bg, bbox, padding=0.05)
    
    # Step 5: Resize to target size
    final = resize_image(cropped, target_size=(224, 224))
    
    return final

## 5. Load EfficientNet Model for Feature Extraction

In [ ]:
import torchvision.models as models
import torchvision.transforms as transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load pretrained EfficientNet-B0
model = models.efficientnet_b0(weights='IMAGENET1K_V1')
model = torch.nn.Sequential(*list(model.children())[:-1])
model = model.to(device)
model.eval()

# Preprocessing transforms
preprocess_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("EfficientNet-B0 loaded successfully")

## 6. Feature Extraction Function

In [ ]:
def extract_features(image):
    """Extract features using EfficientNet"""
    img_tensor = preprocess_transform(image).unsqueeze(0).to(device)
    
    with torch.no_grad():
        features = model(img_tensor)
        features = features.squeeze().cpu().numpy()
    
    return features

## 7. Process Training Data

In [ ]:
X_train = []
y_jenis = []
y_warna = []

print("Processing training images...")
for idx, row in tqdm(train_df.iterrows(), total=len(train_df)):
    img_path = os.path.join(TRAIN_DIR, f"{row['id']}.jpg")
    
    if os.path.exists(img_path):
        try:
            # Preprocess image
            processed_img = preprocess_pipeline(img_path)
            
            # Extract features
            features = extract_features(processed_img)
            
            X_train.append(features)
            y_jenis.append(row['jenis'])
            y_warna.append(row['warna'])
        except Exception as e:
            print(f"Error processing {img_path}: {e}")
            continue

X_train = np.array(X_train)
y_jenis = np.array(y_jenis)
y_warna = np.array(y_warna)

print(f"\nFeatures shape: {X_train.shape}")
print(f"Jenis labels: {y_jenis.shape}")
print(f"Warna labels: {y_warna.shape}")

## 8. Train Classification Models with Ensemble

In [ ]:
from sklearn.ensemble import VotingClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

# Split data for validation
X_train_split, X_val, y_jenis_train, y_jenis_val, y_warna_train, y_warna_val = train_test_split(
    X_train, y_jenis, y_warna, test_size=0.2, random_state=42
)

print("="*60)
print("Training Jenis Classifiers")
print("="*60)

# Individual classifiers for Jenis
xgb_jenis = xgb.XGBClassifier(n_estimators=200, max_depth=8, learning_rate=0.1, random_state=42, n_jobs=-1)
rf_jenis = RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1)
gb_jenis = GradientBoostingClassifier(n_estimators=150, max_depth=6, learning_rate=0.1, random_state=42)
lr_jenis = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)

# Train individual models
print("Training XGBoost...")
xgb_jenis.fit(X_train_split, y_jenis_train)
xgb_acc = accuracy_score(y_jenis_val, xgb_jenis.predict(X_val))
print(f"XGBoost accuracy: {xgb_acc:.4f}")

print("Training Random Forest...")
rf_jenis.fit(X_train_split, y_jenis_train)
rf_acc = accuracy_score(y_jenis_val, rf_jenis.predict(X_val))
print(f"Random Forest accuracy: {rf_acc:.4f}")

print("Training Gradient Boosting...")
gb_jenis.fit(X_train_split, y_jenis_train)
gb_acc = accuracy_score(y_jenis_val, gb_jenis.predict(X_val))
print(f"Gradient Boosting accuracy: {gb_acc:.4f}")

print("Training Logistic Regression...")
lr_jenis.fit(X_train_split, y_jenis_train)
lr_acc = accuracy_score(y_jenis_val, lr_jenis.predict(X_val))
print(f"Logistic Regression accuracy: {lr_acc:.4f}")

# Ensemble for Jenis
ensemble_jenis = VotingClassifier(
    estimators=[
        ('xgb', xgb_jenis),
        ('rf', rf_jenis),
        ('gb', gb_jenis),
        ('lr', lr_jenis)
    ],
    voting='soft',
    n_jobs=-1
)

print("\nTraining Ensemble Jenis...")
ensemble_jenis.fit(X_train_split, y_jenis_train)
ensemble_jenis_acc = accuracy_score(y_jenis_val, ensemble_jenis.predict(X_val))
print(f"Ensemble Jenis accuracy: {ensemble_jenis_acc:.4f}")

print("\n" + "="*60)
print("Training Warna Classifiers")
print("="*60)

# Individual classifiers for Warna
xgb_warna = xgb.XGBClassifier(n_estimators=200, max_depth=8, learning_rate=0.1, random_state=42, n_jobs=-1)
rf_warna = RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1)
gb_warna = GradientBoostingClassifier(n_estimators=150, max_depth=6, learning_rate=0.1, random_state=42)
lr_warna = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)

# Train individual models
print("Training XGBoost...")
xgb_warna.fit(X_train_split, y_warna_train)
xgb_w_acc = accuracy_score(y_warna_val, xgb_warna.predict(X_val))
print(f"XGBoost accuracy: {xgb_w_acc:.4f}")

print("Training Random Forest...")
rf_warna.fit(X_train_split, y_warna_train)
rf_w_acc = accuracy_score(y_warna_val, rf_warna.predict(X_val))
print(f"Random Forest accuracy: {rf_w_acc:.4f}")

print("Training Gradient Boosting...")
gb_warna.fit(X_train_split, y_warna_train)
gb_w_acc = accuracy_score(y_warna_val, gb_warna.predict(X_val))
print(f"Gradient Boosting accuracy: {gb_w_acc:.4f}")

print("Training Logistic Regression...")
lr_warna.fit(X_train_split, y_warna_train)
lr_w_acc = accuracy_score(y_warna_val, lr_warna.predict(X_val))
print(f"Logistic Regression accuracy: {lr_w_acc:.4f}")

# Ensemble for Warna
ensemble_warna = VotingClassifier(
    estimators=[
        ('xgb', xgb_warna),
        ('rf', rf_warna),
        ('gb', gb_warna),
        ('lr', lr_warna)
    ],
    voting='soft',
    n_jobs=-1
)

print("\nTraining Ensemble Warna...")
ensemble_warna.fit(X_train_split, y_warna_train)
ensemble_warna_acc = accuracy_score(y_warna_val, ensemble_warna.predict(X_val))
print(f"Ensemble Warna accuracy: {ensemble_warna_acc:.4f}")

# Retrain on full data
print("\n" + "="*60)
print("Retraining Ensemble on Full Training Data")
print("="*60)

ensemble_jenis.fit(X_train, y_jenis)
ensemble_warna.fit(X_train, y_warna)
print("Training complete")

## 9. Process Test Data and Generate Predictions

In [ ]:
X_test = []
test_ids = []

print("Processing test images...")
for idx, row in tqdm(sample_submission.iterrows(), total=len(sample_submission)):
    img_path = os.path.join(TEST_DIR, f"{row['id']}.jpg")
    
    if os.path.exists(img_path):
        try:
            # Preprocess image
            processed_img = preprocess_pipeline(img_path)
            
            # Extract features
            features = extract_features(processed_img)
            
            X_test.append(features)
            test_ids.append(row['id'])
        except Exception as e:
            print(f"Error processing {img_path}: {e}")
            X_test.append(np.zeros(X_train.shape[1]))
            test_ids.append(row['id'])

X_test = np.array(X_test)

# Generate predictions using ensemble
print("\nGenerating predictions with ensemble...")
pred_jenis = ensemble_jenis.predict(X_test)
pred_warna = ensemble_warna.predict(X_test)

# Create submission
submission = pd.DataFrame({
    'id': test_ids,
    'jenis': pred_jenis,
    'warna': pred_warna
})

submission.to_csv('submission.csv', index=False)
print("\nSubmission saved to submission.csv")
print(f"Predictions generated for {len(submission)} images")

## 10. Visualize Sample Results

In [ ]:
sample_ids = [1, 50, 100]

fig, axes = plt.subplots(len(sample_ids), 5, figsize=(20, 4*len(sample_ids)))

for i, img_id in enumerate(sample_ids):
    img_path = os.path.join(TRAIN_DIR, f"{img_id}.jpg")
    
    if os.path.exists(img_path):
        # Original
        original = Image.open(img_path).convert('RGB')
        axes[i, 0].imshow(original)
        axes[i, 0].set_title('Original')
        axes[i, 0].axis('off')
        
        # CLAHE
        clahe_img = apply_clahe(original)
        axes[i, 1].imshow(clahe_img)
        axes[i, 1].set_title('CLAHE')
        axes[i, 1].axis('off')
        
        # Remove BG
        no_bg = remove_background(clahe_img)
        axes[i, 2].imshow(no_bg)
        axes[i, 2].set_title('Remove BG')
        axes[i, 2].axis('off')
        
        # Cropped
        bbox = get_bounding_box(no_bg)
        cropped = crop_with_padding(no_bg, bbox)
        axes[i, 3].imshow(cropped)
        axes[i, 3].set_title('Cropped')
        axes[i, 3].axis('off')
        
        # Final
        final = resize_image(cropped)
        axes[i, 4].imshow(final)
        axes[i, 4].set_title('Final (224x224)')
        axes[i, 4].axis('off')

plt.tight_layout()
plt.savefig('preprocessing_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()

print("Visualization saved as preprocessing_pipeline.png")

## 11. Pipeline Summary

**Preprocessing Steps:**
1. CLAHE - Adaptive contrast enhancement on L channel
2. RemBG - Background removal using rembg library
3. Localization - Bounding box detection from alpha channel
4. Cropping - Crop with 5% padding
5. Resize - 224x224 with aspect ratio maintained

**Feature Extraction:**
- Model: EfficientNet-B0 (pretrained torchvision)
- Output: 1280-dimensional feature vector

**Classification:**
- Algorithm: Ensemble (Voting Classifier)
- Base Models:
  - XGBoost (n_estimators=200, max_depth=8)
  - Random Forest (n_estimators=200, max_depth=12)
  - Gradient Boosting (n_estimators=150, max_depth=6)
  - Logistic Regression (max_iter=1000)
- Voting: Soft voting (probability-based)
- Tasks: Multi-label (Jenis + Warna)
- Validation split: 80-20